# R5 学术写作 IMRaD · 牛津 Tutorial LLM 仿真 (v6.0)

## Persona (Oxford Tutorial Fellow + HBS Devil's Advocate)

> You are an Oxford tutorial fellow in 学术写作IMRaD (IMRaD structure, APA citation, preregistration, OSF, reproducible research, peer review).
>
> **Never give direct answers.** Use Socratic questioning to make the student reason through IMRaD structure, APA 7th edition statistics, NSW causal inference data, and LLM-as-a-judge peer review.
>
> Act as an HBS devil's advocate: challenge vague claims, demand evidence, poke holes in argument paths. Reject generic templates; force the student to cite real datasets (arxiv / statsmodels / scipy.stats / causaldata NSW / Zheng et al. NeurIPS 2023 / DeepSeek / 天道推演).
>
> End each turn with a probing question. The student must speak 80% of the words; you speak 20%.
>
> Alignment: this tutorial trains ILO-1 (IMRaD structure), ILO-2 (argument path), ILO-3 (APA stats), ILO-5 (peer review), ILO-6 (preregistration/OSF).


## Pre-Tutorial Task (Forced Retrieval, BEFORE the tutorial)

You MUST submit the following BEFORE the tutorial begins. No submission, no tutorial.

1. **Essay (>=300 字)**: Download the ReAct paper (Yao et al., 2022) abstract via `arxiv` (lukasschwab/arxiv.py). Classify each sentence into I/M/R/D. Justify each label in 1 line. Identify the funnel stages present in the Introduction.

2. **Statistical Report**: Run `statsmodels.stats.weightstats.ttest_ind` on `causaldata` NSW data (`re78` for `treat==1` vs `treat==0`, N=445). Write the APA 7th edition sentence including `t(df)`, `p`, Cohen's `d` (with small/medium/large interpretation, Cohen 1988), and 95% CI.

3. **Argument Path**: Use 天道推演 (causal chain -> sandbox -> optimal path) to draft 2 candidate argument paths for an "AI Marketing Agent effectiveness" paper. Predict 1 reviewer objection per path.

4. **Preregistration plan (1 段)**: If you were to preregister this NSW-style study on OSF, name the 3 things you would lock BEFORE data collection (to defend against p-hacking and HARKing).

> Bring these as `pre_tutorial.md`. The tutor will read it cold; the Socratic loop will probe the weakest claim.


In [ ]:
# Multi-turn Socratic loop (>=4 turns, STATIC if/else simulating Socratic follow-up)
# NO real openai/anthropic API calls. The "tutor" is a deterministic state machine
# that branches on student input keywords and always ends with a probing question.
#
# Socratic questions embedded (>=5): 为什么 / 反例 / 若前提变 / 凭什么 / 如何 / 依据
# Each turn forces the student to confront a specific IMRaD / APA / preregistration weakness.

import json, os

STUDENT_INPUTS = [
    "ReAct 摘要第1句是 Introduction 因为说了 We propose ReAct",
    "t(443)=2.84, p<.01, d=0.27",
    "我的论证路径是先理论后实证, 因为这样更有逻辑",
    "preregistration 就是提前写好假设",
]

SOCRATIC_TURNS = [
    # Turn 1: probe Introduction classification
    {
        "trigger_keywords": ["propose", "Introduction", "第1句"],
        "tutor_response": (
            "你标 Introduction 的依据是动词 'propose'。但贡献声明有时也用 'we propose' -- "
            "Introduction 与 Contribution 的边界在哪? **为什么** 'We propose' 一定属于背景而非贡献? "
            "给一个 **反例**: 哪种 'propose' 句应标 Contribution? "
            "**若前提变**为摘要第 5 句才出现 'propose', 你的判断会变吗?"
        ),
        "skill_targeted": "ILO-1 IMRaD structure",
    },
    # Turn 2: probe APA 7th edition precision
    {
        "trigger_keywords": ["t(", "p<", "d="],
        "tutor_response": (
            "你写 `p<.01` 但 p=0.005。**凭什么** 不写精确值 p=.005? APA 第7版对 p≥.001 的要求是什么? "
            "**如何** 报告 95% CI? 你的句子里有 CI 吗? "
            "效应量 d=0.27 是小/中/大? **依据** 是谁 (哪一年) 的标准?"
        ),
        "skill_targeted": "ILO-3 APA statistics",
    },
    # Turn 3: probe argument path (天道推演)
    {
        "trigger_keywords": ["先理论", "先实证", "论证路径", "更有逻辑"],
        "tutor_response": (
            "'更有逻辑' 是模糊断言。**为什么** 先理论比先实证更有说服力? "
            "审稿人会 **如何** 质疑这条路径 (推演第1层)? "
            "你 **凭什么** 不比较另一条路径 (如交替式)? "
            "天道推演要求沙盘展开 3 条路径 x 3 层 -- 你只给了 1 条, **反例** 路径在哪?"
        ),
        "skill_targeted": "ILO-2 argument path",
    },
    # Turn 4: probe preregistration / OSF depth
    {
        "trigger_keywords": ["preregistration", "提前写好", "假设"],
        "tutor_response": (
            "Preregistration 不只是'提前写好假设'。**若前提变**为你在分析阶段发现一个有趣的子组效应, "
            "未预注册的分析算什么 (p-hacking 还是 HARKing)? **为什么**? "
            "OSF 上预注册要锁哪 3 项 (假设/分析计划/代码)? **依据** 是哪份指南? "
            "**如何** 让审稿人相信你的 NSW 结果不是 p-hacking 产物?"
        ),
        "skill_targeted": "ILO-6 preregistration / OSF",
    },
    # Turn 5: probe peer review (LLM-as-a-judge bias)
    {
        "trigger_keywords": ["peer review", "judge", "reviewer", "审稿"],
        "tutor_response": (
            "你打算用 LLM-as-a-judge 评估自己的论文。**为什么** 这有自我偏好偏差 (Zheng et al., NeurIPS 2023, arXiv 2306.05685)? "
            "**如何** 用 DeepSeek 多 judge 投票缓解? "
            "**反例**: LLM judge 给所有部分同 4/5 分, 这是哪类偏差? "
            "它对应因果阶梯 L1 还是 L2? **凭什么**?"
        ),
        "skill_targeted": "ILO-5 peer review / LLM-as-a-judge",
    },
]

def run_socratic_loop(student_inputs, max_turns=5):
    """Static if/else Socratic simulator. No real LLM API call."""
    log = []
    for i, student_text in enumerate(student_inputs[:max_turns]):
        matched = None
        for turn in SOCRATIC_TURNS:
            if any(kw.lower() in student_text.lower() for kw in turn["trigger_keywords"]):
                matched = turn
                break
        if matched is None:
            matched = SOCRATIC_TURNS[i % len(SOCRATIC_TURNS)]
        log.append({
            "turn": i + 1,
            "student_input": student_text,
            "tutor_response": matched["tutor_response"],
            "skill_targeted": matched["skill_targeted"],
        })
    return log

# Run the loop with the static student inputs (simulating a 5-turn tutorial)
tutorial_log = run_socratic_loop(STUDENT_INPUTS, max_turns=5)

# Pretty print
for entry in tutorial_log:
    print(f"--- Turn {entry['turn']} ({entry['skill_targeted']}) ---")
    print(f"Student: {entry['student_input']}")
    print(f"Tutor  : {entry['tutor_response']}")
    print()

# Count Socratic question keywords to verify >=5
import re
all_tutor_text = " ".join(e["tutor_response"] for e in tutorial_log)
soc_q = len(re.findall(r"(?im)(why|为什么|如何|how could|what if|若|反例|counterexample|凭什么|依据|假设.*变)", all_tutor_text))
print(f"\n[Verify] Socratic question keywords hit: {soc_q} (need >=5)")
assert soc_q >= 5, "Not enough Socratic questions"
assert len(tutorial_log) >= 4, "Need >=4 turns"
print(f"[Verify] Turns: {len(tutorial_log)} (need >=4)")
print("[Verify] Socratic loop OK (static, no real LLM API call)")


In [ ]:
# student_model.json: read/write mastery state and blind spots
# Records per-ILO mastery (0.0-1.0), blind spots list, and last tutorial turn.

import json, os

SM_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(SM_PATH):
        with open(SM_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "R5",
        "ilo_mastery": {
            "ILO-1_IMRaD_structure": 0.0,
            "ILO-2_argument_path": 0.0,
            "ILO-3_APA_stats": 0.0,
            "ILO-4_title_abstract": 0.0,
            "ILO-5_peer_review": 0.0,
            "ILO-6_preregistration_OSF": 0.0,
        },
        "blind_spots": [],
        "tutorials_completed": 0,
        "last_turn": 0,
    }

def save_student_model(model):
    with open(SM_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

# Load existing (or init)
sm = load_student_model()
print("Loaded student_model:")
print(json.dumps(sm, ensure_ascii=False, indent=2))

# Update mastery based on the tutorial log (heuristic: each matched turn = +0.15 mastery)
skill_to_ilo = {
    "ILO-1 IMRaD structure": "ILO-1_IMRaD_structure",
    "ILO-2 argument path": "ILO-2_argument_path",
    "ILO-3 APA statistics": "ILO-3_APA_stats",
    "ILO-5 peer review / LLM-as-a-judge": "ILO-5_peer_review",
    "ILO-6 preregistration / OSF": "ILO-6_preregistration_OSF",
}

for entry in tutorial_log:
    ilo_key = skill_to_ilo.get(entry["skill_targeted"])
    if ilo_key and ilo_key in sm["ilo_mastery"]:
        sm["ilo_mastery"][ilo_key] = min(1.0, sm["ilo_mastery"][ilo_key] + 0.15)

# Identify blind spots: any ILO below 0.6 after tutorial
sm["blind_spots"] = [ilo for ilo, m in sm["ilo_mastery"].items() if m < 0.6]
sm["tutorials_completed"] += 1
sm["last_turn"] = len(tutorial_log)

save_student_model(sm)
print("\nUpdated student_model (saved):")
print(json.dumps(sm, ensure_ascii=False, indent=2))


## Hattie 4-Level Formative Feedback (Hattie & Timperley, 2007)

The tutor delivers feedback at 4 levels. Self-level praise is intentionally avoided (Hattie: Self-level feedback has low effect size, d=0.20; Task/Process/Feed-Forward have higher).

### [TASK] Task-level (about the specific IMRaD sentence / APA number)
- **Feedback**: "Your label 'Introduction' for ReAct sentence 1 is plausible but you cited only the verb 'propose'. The TASK-level criterion is: does the sentence establish background OR contribution? Re-read: 'We propose ReAct, a general paradigm...' -- this is contribution, not background. Re-label as Contribution (still part of Introduction, but a different funnel stage)."
- **Effect size target**: d > 0.50 (Task-level)

### [PROCESS] Process-level (about the reasoning strategy)
- **Feedback**: "Your PROCESS for APA reporting skipped CI. The strategy is: (1) run ttest_ind -> get t/p, (2) compute Cohen's d, (3) compute 95% CI via scipy.stats.t, (4) assemble template `t(df)=X.XX, p=.XXX, d=X.XX, 95% CI [LL,UL]`, (5) interpret effect size. You jumped from (1) to (4) skipping (3). Next time, use a 5-step checklist BEFORE writing the sentence."
- **Effect size target**: d > 0.60 (Process-level)

### [SELF-REG] Self-regulation-level (about student self-monitoring)
- **Feedback**: "You wrote '更有逻辑' without evidence. This signals a SELF-REG gap: you did not pause to ask 'how would a reviewer challenge this?'. Self-regulation move: before any evaluative claim, force yourself to write 1 counter-argument. If you cannot, the claim is ungrounded. Track this in your blind_spots list."
- **Effect size target**: d > 0.50 (Self-Reg)

### [FEED-FORWARD] Feed-forward-level (what to do NEXT, not what was wrong)
- **Feedback**: "For your NEXT submission (progressive_project milestone Day 7), do NOT re-write the Introduction. Instead, complete drill A2 (天道推演 3-path sandbox) in practice.md -- produce 3 argument paths with 3-layer推演 each. Bring the strongest path to the next tutorial. This is feed-forward: it redirects effort to the highest-leverage gap (argument path depth), not the lowest (sentence labeling)."
- **Effect size target**: d > 0.70 (Feed-Forward, highest)

> Hattie note: Self-level praise ("Good job!") is intentionally absent. It inflates self-concept without raising competence. We give Task/Process/Self-Reg/Feed-Forward only.


## 限频 (Usage Limit, prevent dependency)

- **每单元 1 次/天**: 本 Oxford tutorial LLM 仿真每天最多运行 1 次。`student_model.json` 的 `tutorials_completed` 字段会强制计数。
- **为什么限频**: 防止学生依赖 LLM 推理而非自己检索 (retrieval practice 优于重读; spaced retrieval 优于 massed)。Hattie 研究表明, 过度反馈反而降低 self-regulation。
- **超限处理**: 若当日已运行, 系统返回 "今日 tutorial 已用完。请先做 schedule.json 的 due card 间隔重复, 明日再来。"

## Exit Artifact (tutorial 结束必交)

完成本 tutorial 后, 学生必须提交 `exit_artifact.md`:

1. **2-3 个盲点** (从 `student_model.json` 的 `blind_spots` 提取, 自行扩展说明):
   - 例: "ILO-3 APA stats: 我仍不确定 p=.005 vs p<.01 的边界"
   - 例: "ILO-6 preregistration: 我分不清 p-hacking 与 HARKing 的实操差异"
   - 例: "ILO-2 argument path: 我的 3 条路径都偏同质, 沙盘未真正展开"

2. **推荐复习单元** (基于盲点 cross-link):
   - ILO-3 弱 -> 复习 `module-r-research-methodology/day-r3-...` (混合方法统计) + schedule.json card C3 (APA 格式)
   - ILO-6 弱 -> 复习 `module-r-research-methodology/day-r6-research-ethics` (研究伦理与 AI 治理, preregistration 是伦理前置) + schedule.json card C8
   - ILO-2 弱 -> 复习 `module-r-research-methodology/day-r4-prisma` (PRISMA 系统综述, 论证路径基础) + practice.md drill A2

3. **下一次 tutorial 的 1 个聚焦问题** (学生自拟, tutor 下次优先追问):
   - 例: "如何在 Discussion 中诚实报告 NSW 效应量 d=0.27 (small) 而不让审稿人觉得研究无价值?"

> 限频 + exit artifact 共同保证: tutorial 不是"问 LLM 要答案", 而是"逼自己暴露盲点 + 主动规划下次学习"。
